In [22]:
import pandas as pd
import numpy as np
import hashlib

# Load hospital_1.csv from depression dataset split
df = pd.read_csv('hospital_1.csv')
df.head()


,full_name,email,education,urban,gender,engnat,screensize,uniquenetworklocation,hand,religion,...,TIPI1,TIPI2,TIPI3,TIPI4,TIPI5,TIPI6,TIPI7,TIPI8,TIPI9,TIPI10
0,Julie Miller,liutracy@example.com,University degree,Urban,Female,No,Small screen,Unique network,Right,Muslim,...,4,6,2,7,4,6,4,7,2,7
1,Amy Serrano,jason26@example.net,High school,Suburban,Male,No,Small screen,Shared network,Right,Muslim,...,3,4,1,7,1,7,7,7,3,7
2,David Farrell,kevinwebb@example.com,High school,Suburban,Female,Yes,Small screen,Unique network,Right,Christian (Catholic),...,4,4,1,1,1,1,4,4,3,3
3,Albert Freeman,laurabrooks@example.com,Less than high school,Suburban,Male,No,Small screen,Unique network,Right,Christian (Catholic),...,7,5,4,4,5,5,5,5,5,4
4,Nicole Pittman,tammy83@example.com,High school,Urban,Female,No,Big screen,Unique network,Right,Christian (Mormon),...,6,6,7,7,4,7,7,5,2,2


In [23]:
# Drop known personal identifiers or fingerprinting risks
cols_to_drop = ['country','email','full_name']
df = df.drop(columns=cols_to_drop, errors='ignore')
print("Shape after dropping personal identifiers:", df.shape)

Shape after dropping personal identifiers: (12874, 38)


In [24]:
# Generalize age into categories like 'Child', 'Teen', 'Adult', 'Senior'
def generalize_age(age_val):
    try:
        age = float(age_val)
        if age < 13:
            return 'Child'
        elif 13 <= age < 20:
            return 'Teen'
        elif 20 <= age < 60:
            return 'Adult'
        elif age >= 60:
            return 'Senior'
        else:
            return np.nan
    except:
        return np.nan

# Apply directly in the same column
df['age'] = df['age'].apply(generalize_age)

# Drop rows with invalid or unparseable age
df = df[df['age'].notna()]



In [25]:
# Identify columns to tokenize
columns_to_tokenise = df.select_dtypes(include='object').columns.difference(['gender', 'engnat','age'])

# Define tokenizer
def tokenise(val):
    return 'TKN_' + hashlib.md5(str(val).encode()).hexdigest()[:8]

# Apply tokenisation
for col in columns_to_tokenise:
    df[col] = df[col].astype(str).apply(tokenise)

In [26]:
# DASS question codes for Depression
depression_qs = ['Q3A', 'Q5A', 'Q10A', 'Q13A', 'Q16A', 'Q17A', 'Q21A', 'Q24A', 'Q26A', 'Q31A', 'Q34A', 'Q37A', 'Q38A', 'Q42A']
# Only pick those depression question columns from df
depression_df = df[depression_qs].copy()

# Convert to numeric (in case some cells are strings), and subtract 1
depression_df = depression_df.apply(pd.to_numeric, errors='coerce') - 1
depression_df['Total_Count'] = depression_df.sum(axis=1)
def get_condition(score):
    if score <= 9:
        return 'Normal'
    elif 10 <= score <= 13:
        return 'Mild'
    elif 14 <= score <= 20:
        return 'Moderate'
    elif 21 <= score <= 27:
        return 'Severe'
    else:
        return 'Extremely Severe'

depression_df['Condition'] = depression_df['Total_Count'].apply(get_condition)
# Now add the 'Condition' column to your main dataframe
df['Condition'] = depression_df['Condition']


In [27]:
k = 3

# Define quasi-identifiers
qi_cols = ['gender', 'education', 'urban', 'age', 'race', 'religion']

# Convert all to string to avoid grouping errors
for col in qi_cols:
    df[col] = df[col].astype(str)

# Group and filter based on k
grouped = df.groupby(qi_cols)
valid_indices = []

for _, group in grouped:
    if len(group) >= k:
        valid_indices.extend(group.index)

df_k_anon = df.loc[valid_indices].reset_index(drop=True)
print(f"Shape after K-Anonymity: {df_k_anon.shape}")


Shape after K-Anonymity: (12038, 39)


In [28]:
l = 2
sensitive_col = 'orientation'

# Group on QIs
grouped = df_k_anon.groupby(qi_cols)
valid_indices_ldiv = []

for _, group in grouped:
    if group[sensitive_col].nunique() >= l:
        valid_indices_ldiv.extend(group.index)

df_ldiverse = df_k_anon.loc[valid_indices_ldiv].reset_index(drop=True)
print(f"Shape after L-Diversity: {df_ldiverse.shape}")

df_ldiverse.to_csv('Final_Depression1_anonymised.csv', index=False)
print("Final anonymised dataset saved as Final_Depression1_anonymised.csv")

Shape after L-Diversity: (11670, 39)
Final anonymised dataset saved as Final_Depression1_anonymised.csv
